# 🐆 Pumas, Guanacos e Ovelhas na Patagônia
## Modelagem e simulação numérica de uma rede trófica (1 predador, 2 presas)

**Notebook didático** — método de **Euler** aplicado a um sistema de equações
diferenciais ordinárias (EDOs) acopladas.

> Inspirado no artigo *"Coexistence of pumas, guanacos and sheep in Patagonia"*
> ([arXiv:2412.02936](https://arxiv.org/html/2412.02936v1)).

---

### O que você vai aprender aqui
1. **O problema ecológico** — por que pumas, guanacos e ovelhas formam um caso clássico de conflito ecológico e econômico.
2. **Como traduzir biologia em equações** — construímos o modelo *termo por termo*.
3. **O método de Euler** — o jeito mais simples de resolver uma EDO no computador, deduzido do zero.
4. **Simulação e interpretação** — rodamos, plotamos e lemos os resultados.
5. **Erro e estabilidade** — por que o passo `Δt` importa, comparando com um integrador de alta ordem.
6. **Cenários** — coexistência, dominância das ovelhas e o efeito de caçar pumas.

Há ainda um **simulador web 3D interativo** (`app.py`) que usa exatamente o mesmo modelo.


## 1. O problema: a estepe patagônica

Na Patagônia convivem três personagens:

| Espécie | Papel | Observação |
|---------|-------|------------|
| 🐆 **Puma** (*Puma concolor*) | **Predador** | Caça tanto guanacos quanto ovelhas. |
| 🦙 **Guanaco** (*Lama guanicoe*) | **Presa nativa** | Presa natural do puma na estepe. |
| 🐑 **Ovelha** (*Ovis aries*) | **Presa introduzida** | Trazida por colonizadores europeus; base da pecuária. |

A rede trófica tem o formato de **um predador e duas presas que competem entre si**:

```
              🐆 PUMA
             /        \
        caça /          \ caça
            v            v
     🦙 GUANACO  <----->  🐑 OVELHA
              competição
           (pasto e espaço)
```

**Tensões do sistema**
- As **ovelhas** são presas mais fáceis (vulneráveis) e economicamente valiosas → atrito com fazendeiros.
- As **ovelhas competem** com os guanacos por vegetação e espaço.
- **Caçar pumas** (para proteger o rebanho) pode *desestabilizar* o sistema: sem predador, a presa que cresce mais rápido explode até a capacidade de suporte.

A pergunta de fundo: **é possível as três espécies coexistirem?** As equações ajudam a responder.


## 2. Das ideias biológicas às equações

Vamos chamar as populações de:

- $P$ = pumas (predador)
- $G$ = guanacos (presa nativa)
- $O$ = ovelhas (presa introduzida)

Uma **equação diferencial** descreve a *taxa de variação* de cada população,
$\frac{dP}{dt}$, $\frac{dG}{dt}$, $\frac{dO}{dt}$ — isto é, *quanto a população muda por unidade de tempo*.
Construímos cada uma somando os efeitos biológicos, um de cada vez.

### 2.1 Crescimento logístico das presas
Sozinha e com recursos limitados, uma presa cresce de forma **logística**:

$$\frac{dG}{dt} = r_G\,G\left(1 - \frac{G}{K_G}\right)$$

- $r_G$: taxa de crescimento intrínseco (quão rápido se reproduz).
- $K_G$: **capacidade de suporte** — população máxima que o ambiente sustenta.
- Quando $G \ll K_G$, cresce quase exponencialmente; quando $G \to K_G$, o crescimento → 0.

### 2.2 Predação (o puma comendo a presa)
A predação remove presas a uma taxa proporcional aos *encontros* entre predador e presa ($\propto P\cdot G$):

$$-\,a_G\,P\,G$$

- $a_G$: **taxa de ataque** do puma ao guanaco. As ovelhas costumam ter $a_O$ maior (mais vulneráveis).

### 2.3 Competição entre as presas
Guanacos e ovelhas disputam pasto e espaço. Cada espécie reduz o crescimento da outra ($\propto G\cdot O$):

$$-\,c_{GO}\,G\,O \quad(\text{efeito das ovelhas sobre os guanacos})$$

### 2.4 A equação do predador
O puma **morre** a uma taxa natural $m_P$ e **ganha energia** (e se reproduz) ao caçar — proporcional ao que come:

$$\frac{dP}{dt} = -\,m_P\,P + e_G\,P\,G + e_O\,P\,O$$

- $e_G, e_O$: **eficiência de conversão** — quanto cada presa caçada se converte em novos pumas.


### 2.5 O sistema completo

Juntando tudo, chegamos às **três EDOs acopladas**:

$$\boxed{\;\frac{dP}{dt} = -\,m_P\,P + e_G\,P\,G + e_O\,P\,O\;}$$

$$\boxed{\;\frac{dG}{dt} = r_G\,G\left(1-\frac{G}{K_G}\right) - a_G\,P\,G - c_{GO}\,G\,O\;}$$

$$\boxed{\;\frac{dO}{dt} = r_O\,O\left(1-\frac{O}{K_O}\right) - a_O\,P\,O - c_{OG}\,O\,G\;}$$

> ⚠️ **Correção de digitação.** O enunciado original trazia o termo logístico das ovelhas
> como $\left(1 - \frac{K_O}{K_O}\right)$, que é sempre **zero**. O correto é
> $\left(1 - \frac{O}{K_O}\right)$ — e é o que usamos.

São equações *acopladas* (cada uma depende das outras) e *não lineares* (têm produtos
$PG$, $GO$, ...). **Não há fórmula fechada** para a solução → precisamos de
**métodos numéricos**.


### 2.6 Dicionário de parâmetros

| Parâmetro | Significado | Espécie |
|-----------|-------------|---------|
| $m_P$ | mortalidade natural do puma | 🐆 |
| $e_G,\ e_O$ | eficiência de conversão (caça de guanaco / ovelha) | 🐆 |
| $r_G,\ r_O$ | taxa de crescimento intrínseco | 🦙 / 🐑 |
| $K_G,\ K_O$ | capacidade de suporte do ambiente | 🦙 / 🐑 |
| $a_G,\ a_O$ | taxa de ataque do puma | 🦙 / 🐑 |
| $c_{GO}$ | competição: efeito das ovelhas sobre guanacos | 🦙 |
| $c_{OG}$ | competição: efeito dos guanacos sobre ovelhas | 🐑 |


## 3. Preparando o código

Definimos o **campo vetorial** $f$ do sistema — ou seja, uma função que recebe o
estado $(P, G, O)$ e devolve as três derivadas $(\dot P, \dot G, \dot O)$.
Esse é exatamente o código que está em `model.py` e que alimenta o simulador 3D.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

# cores consistentes em todo o notebook
COR = {"P": "#d6452b", "G": "#c98a3a", "O": "#7a8aa0"}


In [ ]:
@dataclass
class Params:
    """Todos os parâmetros do modelo, com valores padrão (regime de coexistência)."""
    # Pumas
    m_P: float = 0.50
    e_G: float = 0.0011
    e_O: float = 0.0009
    # Guanacos
    r_G: float = 1.10
    K_G: float = 900.0
    a_G: float = 0.0045
    c_GO: float = 0.0005
    # Ovelhas
    r_O: float = 1.00
    K_O: float = 900.0
    a_O: float = 0.0055
    c_OG: float = 0.0003


def derivatives(P, G, O, p: Params):
    """Campo vetorial f(P,G,O) = (dP/dt, dG/dt, dO/dt)."""
    dP = -p.m_P * P + p.e_G * P * G + p.e_O * P * O
    dG = p.r_G * G * (1 - G / p.K_G) - p.a_G * P * G - p.c_GO * G * O
    dO = p.r_O * O * (1 - O / p.K_O) - p.a_O * P * O - p.c_OG * O * G
    return dP, dG, dO

print("Parâmetros padrão:")
for k, v in asdict(Params()).items():
    print(f"  {k:5s} = {v}")


## 4. O método de Euler

### 4.1 A ideia
Uma EDO $\dfrac{dy}{dt} = f(y)$ nos diz a **inclinação** da solução em cada ponto.
A *definição* de derivada é

$$\frac{dy}{dt} \approx \frac{y(t+\Delta t) - y(t)}{\Delta t}.$$

Isolando $y(t+\Delta t)$:

$$\boxed{\; y_{n+1} = y_n + \Delta t \cdot f(y_n) \;}$$

**Interpretação geométrica:** estando em $y_n$, calculamos a inclinação $f(y_n)$ ali e
**andamos em linha reta** por um passo de tempo $\Delta t$ nessa direção. Repetindo,
construímos a curva inteira como uma sucessão de pequenos segmentos de reta.

### 4.2 Para o nosso sistema (3 variáveis)
A cada passo atualizamos as três populações **simultaneamente**, usando os valores do
passo anterior:

$$
\begin{aligned}
P_{n+1} &= P_n + \Delta t\,\big(-m_P P_n + e_G P_n G_n + e_O P_n O_n\big)\\
G_{n+1} &= G_n + \Delta t\,\big(r_G G_n(1 - G_n/K_G) - a_G P_n G_n - c_{GO} G_n O_n\big)\\
O_{n+1} &= O_n + \Delta t\,\big(r_O O_n(1 - O_n/K_O) - a_O P_n O_n - c_{OG} O_n G_n\big)
\end{aligned}
$$

### 4.3 Erro
- **Erro local** (por passo): $\mathcal{O}(\Delta t^2)$.
- **Erro global** (acumulado): $\mathcal{O}(\Delta t)$ → Euler é um método de **1ª ordem**.
  Reduzir $\Delta t$ pela metade reduz o erro pela metade (aproximadamente).

### 4.4 Pseudocódigo
```
estado  <- (P0, G0, O0)
para cada passo n:
    (dP, dG, dO) <- f(estado)
    estado <- estado + Δt · (dP, dG, dO)
    guarde estado
```


In [ ]:
def euler_step(state, p: Params, dt: float):
    """Um passo de Euler. state = (P, G, O)."""
    P, G, O = state
    dP, dG, dO = derivatives(P, G, O, p)
    P, G, O = P + dt * dP, G + dt * dG, O + dt * dO
    # populações não podem ser negativas
    return max(P, 0.0), max(G, 0.0), max(O, 0.0)


def simulate(p: Params, P0=40.0, G0=200.0, O0=150.0, dt=0.02, t_final=120.0):
    """Integra o sistema pelo método de Euler. Retorna arrays (t, P, G, O)."""
    n = int(round(t_final / dt))
    t = np.linspace(0.0, n * dt, n + 1)
    P = np.empty(n + 1); G = np.empty(n + 1); O = np.empty(n + 1)
    P[0], G[0], O[0] = P0, G0, O0
    state = (P0, G0, O0)
    for i in range(n):
        state = euler_step(state, p, dt)
        P[i+1], G[i+1], O[i+1] = state
    return t, P, G, O

# teste rápido
t, P, G, O = simulate(Params())
print(f"{len(t)} passos | estado final: P={P[-1]:.1f}  G={G[-1]:.1f}  O={O[-1]:.1f}")


## 5. Primeira simulação: a dinâmica no tempo

Rodamos com os parâmetros padrão e plotamos as três populações.


In [ ]:
t, P, G, O = simulate(Params(), t_final=120.0)

plt.figure()
plt.plot(t, G, color=COR["G"], lw=2, label="Guanacos $G$")
plt.plot(t, O, color=COR["O"], lw=2, label="Ovelhas $O$")
plt.plot(t, P, color=COR["P"], lw=2, label="Pumas $P$")
plt.xlabel("tempo $t$"); plt.ylabel("população")
plt.title("Dinâmica populacional — método de Euler ($\\Delta t = 0{,}02$)")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Equilíbrio aproximado:  P*≈{P[-1]:.0f}   G*≈{G[-1]:.0f}   O*≈{O[-1]:.0f}")


**Leitura do gráfico.** As populações **oscilam de forma amortecida** e convergem
para um **equilíbrio de coexistência** (todas as três positivas). É o cenário central do
artigo: as três espécies *conseguem* coexistir para esses parâmetros. As oscilações
iniciais são o clássico "sobe-e-desce" predador-presa: muitos guanacos → pumas se
multiplicam → guanacos caem → pumas faltam comida → guanacos se recuperam, e assim por diante,
até o sistema assentar.


## 6. O retrato de fase 3D

Em vez de olhar cada população *contra o tempo*, podemos olhar a **trajetória no espaço
de estados** $(G, O, P)$. Cada ponto é um "instantâneo" do ecossistema; a curva mostra
para onde o sistema caminha. O ponto final é o **equilíbrio** (atrator).


In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
sc = ax.scatter(G, O, P, c=t, cmap="viridis", s=4)
ax.plot(G, O, P, color="gray", lw=0.5, alpha=0.5)
ax.scatter([G[0]],[O[0]],[P[0]], color="black", s=60, label="início")
ax.scatter([G[-1]],[O[-1]],[P[-1]], color="red", s=80, marker="*", label="equilíbrio")
ax.set_xlabel("Guanacos $G$"); ax.set_ylabel("Ovelhas $O$"); ax.set_zlabel("Pumas $P$")
ax.set_title("Retrato de fase 3D")
fig.colorbar(sc, ax=ax, label="tempo $t$", shrink=0.6)
ax.legend(); plt.tight_layout(); plt.show()


A trajetória **espirala para dentro** rumo a um único ponto — a assinatura de um
**equilíbrio estável**. É exatamente o que o simulador 3D `app.py` desenha em tempo real.


## 7. O passo $\Delta t$ importa: precisão e estabilidade

Euler é simples, mas **frágil**: se o passo $\Delta t$ for grande demais, a aproximação
"linha reta" erra muito e a solução pode **oscilar artificialmente** ou até **explodir** —
um erro *numérico*, não biológico. Vamos ver.


In [ ]:
plt.figure(figsize=(11, 5))
for dt in [0.02, 0.15, 0.4, 0.7]:
    t, P, G, O = simulate(Params(), dt=dt, t_final=120.0)
    plt.plot(t, np.clip(G, 0, 2000), lw=1.6, label=f"$\\Delta t={dt}$")
plt.xlabel("tempo $t$"); plt.ylabel("Guanacos $G$ (clip em 2000)")
plt.title("Efeito do passo de tempo na solução de Euler (população de guanacos)")
plt.legend(); plt.tight_layout(); plt.show()


- $\Delta t = 0{,}02$ → curva suave e correta.
- $\Delta t$ maior → oscilações cada vez mais grosseiras; para $\Delta t$ grande o método
  fica **instável** (oscila/diverge) mesmo que a solução verdadeira seja calma.

**Regra prática:** comece pequeno e diminua $\Delta t$ até a solução **parar de mudar**.
No simulador, o slider `Δt` deixa você sentir esse efeito ao vivo.


## 8. Quão bom é o Euler? Comparação com um integrador de alta ordem

Vamos comparar o Euler com o **Runge–Kutta adaptativo (RK45)** do SciPy, usado como
*referência* de alta precisão. Depois medimos o **erro global** do Euler para vários
$\Delta t$ e verificamos a **convergência de 1ª ordem** (erro $\propto \Delta t$).


In [ ]:
from scipy.integrate import solve_ivp

p = Params()
y0 = [40.0, 200.0, 150.0]   # (P, G, O)
t_final = 120.0

def rhs(t, y):
    P, G, O = y
    return list(derivatives(P, G, O, p))

# referência de alta precisão
ref = solve_ivp(rhs, [0, t_final], y0, method="RK45",
                rtol=1e-10, atol=1e-12, dense_output=True)
y_ref_final = ref.y[:, -1]   # (P, G, O) no tempo final

# erro do Euler no tempo final para vários dt
dts = [0.4, 0.2, 0.1, 0.05, 0.025, 0.0125]
erros = []
for dt in dts:
    t, P, G, O = simulate(p, P0=y0[0], G0=y0[1], O0=y0[2], dt=dt, t_final=t_final)
    euler_final = np.array([P[-1], G[-1], O[-1]])
    erros.append(np.linalg.norm(euler_final - y_ref_final))

plt.figure(figsize=(7, 5))
plt.loglog(dts, erros, "o-", color="#d6452b", label="erro do Euler")
plt.loglog(dts, [e*(d/dts[0]) for d, e in zip(dts, [erros[0]]*len(dts))],
           "k--", alpha=0.6, label="inclinação 1 (referência)")
plt.gca().invert_xaxis()
plt.xlabel("passo $\\Delta t$"); plt.ylabel("erro no tempo final (norma)")
plt.title("Convergência do método de Euler (1ª ordem)")
plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.tight_layout(); plt.show()

print("Referência RK45 (tempo final):  P=%.2f  G=%.2f  O=%.2f" % tuple(y_ref_final))
for d, e in zip(dts, erros):
    print(f"  Δt={d:<7} → erro={e:.3f}")


No gráfico log–log, os pontos do erro acompanham a reta de **inclinação 1**: cada vez
que dividimos $\Delta t$ por 2, o erro também cai ~pela metade. Isso é a marca registrada de
um método de **primeira ordem**. Métodos como RK4 têm inclinação 4 (erro $\propto\Delta t^4$),
muito mais precisos por passo — por isso o artigo original usa Runge–Kutta. O Euler vence em
**simplicidade e clareza didática**, que é o nosso objetivo aqui.


## 9. Cenários ecológicos: mudando os parâmetros

O mesmo modelo produz **destinos muito diferentes** conforme os parâmetros. Reproduzimos
abaixo os quatro cenários disponíveis no simulador 3D.


In [ ]:
CENARIOS = {
    "Coexistência": Params(),
    "Ovelhas dominam": Params(m_P=0.40, e_G=0.0010, e_O=0.0011, r_G=0.80, K_G=700,
                              a_G=0.0050, c_GO=0.0012, r_O=0.70, K_O=700,
                              a_O=0.0035, c_OG=0.0003),
    "Caça intensa de pumas": Params(m_P=1.00, e_G=0.0006, e_O=0.0004, c_GO=0.0008, c_OG=0.0004),
    "Sem competição": Params(c_GO=0.0, c_OG=0.0),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (nome, par) in zip(axes.ravel(), CENARIOS.items()):
    t, P, G, O = simulate(par, t_final=150.0)
    ax.plot(t, G, color=COR["G"], lw=2, label="Guanacos")
    ax.plot(t, O, color=COR["O"], lw=2, label="Ovelhas")
    ax.plot(t, P, color=COR["P"], lw=2, label="Pumas")
    ax.set_title(nome); ax.set_xlabel("t"); ax.set_ylabel("população")
    ax.legend(fontsize=9)
plt.suptitle("Quatro cenários do mesmo modelo", fontsize=14)
plt.tight_layout(); plt.show()


**O que observar**
- **Coexistência:** as três espécies se estabilizam juntas.
- **Ovelhas dominam:** com competição forte das ovelhas ($c_{GO}$ alto), os guanacos
  são empurrados para baixo — o **resultado central do artigo** (ovelhas vencem a disputa por pasto).
- **Caça intensa de pumas:** com mortalidade $m_P$ alta, os **pumas se extinguem**; sem
  predador, as presas crescem **até a capacidade de suporte**. É a advertência ecológica:
  remover o predador *desestabiliza* o sistema.
- **Sem competição:** zerando $c_{GO}, c_{OG}$, cada presa só "sente" o predador — dinâmica
  predador-presa mais limpa.


## 10. O simulador 3D interativo

Tudo o que vimos aqui está num **simulador web 3D** (`app.py`, feito com Dash + Plotly):

- os **rebanhos aparecem sobre o relevo** patagônico e crescem/encolhem em tempo real;
- você mexe em **qualquer parâmetro com sliders** e vê o efeito na hora;
- traz **série temporal**, **retrato de fase 3D** e os **cenários** prontos.

Para rodar:

```bash
pip install -r requirements.txt
python app.py
# abra http://127.0.0.1:8050
```

---

## 11. Exercícios

1. **Equilíbrio do predador.** No equilíbrio $\dot P = 0$ com $P>0$, mostre que
   $e_G G^* + e_O O^* = m_P$. Verifique numericamente com os valores impressos na Seção 5.
2. **Limiar de extinção do puma.** Aumente $m_P$ aos poucos. A partir de que valor os pumas
   se extinguem? Relacione com $e_G K_G + e_O K_O$.
3. **Invasão das ovelhas.** Fixe os guanacos e aumente $c_{GO}$. Quando os guanacos colapsam?
4. **Estabilidade numérica.** Ache o maior $\Delta t$ que ainda dá uma solução suave (sem
   oscilação artificial) no cenário padrão.
5. **Euler melhorado.** Implemente o método de Euler *melhorado* (Heun/RK2) e refaça o
   gráfico de convergência da Seção 8. Que inclinação você obtém?

## 12. Referências
- *Coexistence of pumas, guanacos and sheep in Patagonia* — arXiv:2412.02936.
- Lotka, A. J. (1925); Volterra, V. (1926) — equações predador-presa.
- Murray, J. D. *Mathematical Biology* — modelos de competição e predação.
